# M5 · Validación del PR #22 y prueba de caída 

Corre el harness (`m4/harness_m4.py`) con la configuración de la Demo (**fp16**) y guarda todo en `m5/`, sin tocar los resultados históricos de `m4/`.

**Antes de correr, en Kaggle (panel derecho → Settings):**
- *Accelerator* → **GPU T4 x1**
- *Internet* → **On** (la cuenta tiene que estar verificada con teléfono)
- La rama `maria-m5` tiene que estar subida a GitHub con el harness nuevo.

## 1 · Entorno: clonar la rama e instalar

In [ ]:
import os
RAMA = "maria-m5"
if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia
!git log -1 --oneline
!grep -c "TODAS_LAS_CORRIDAS" m4/harness_m4.py   # debe ser mayor que 0: es el harness nuevo

In [ ]:
%pip install -q -r requirements.txt
# Mismo arreglo que en los notebooks de M2-M5: torchao viejo hace tronar a peft al importarse.
!pip uninstall -y -q torchao

import torch, transformers
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__)

## 2 · Prueba de humo (2 casos por corrida, ~5 min)
Solo confirma que todo carga. Va a `m5/humo/`, que no se sube al repo.

In [ ]:
!TRIAGE_CUANT=fp16 python m4/harness_m4.py --salida m5/humo --limite 2

## 3 · Corrida completa: validación del PR #22 (~15-20 min)
con_foto (10) + sin_foto (10) + m3_texto (37). La latencia pareada debe salir con n=9.

In [ ]:
!TRIAGE_CUANT=fp16 python m4/harness_m4.py --salida m5/harness

## 4 · Prueba de caída: BETO y RAG apagados
Los 10 casos sin foto. Todos deben salir `sin_senal`, ninguno `no_urgente`, y la respuesta debe decir «revisión manual» → `"pasa": true`.

In [ ]:
!TRIAGE_CUANT=fp16 python m4/harness_m4.py --salida m5/caida --corridas caida

## 5 · Resultados

In [ ]:
import json
r = json.load(open("m5/harness/resultados.json", encoding="utf-8"))
print("ENTORNO:", json.dumps(r.get("entorno"), ensure_ascii=False, indent=1))
for n, c in r["corridas"].items():
    print(f"\n{n}: recall={c['d3_recall_urgente']}  D1={c['d1_aciertos']}  sin_senal={c['casos_sin_senal']}  "
          f"p50={c['latencia_s_p50']} s  p95={c['latencia_s_p95']} s  rutas={c['rutas_cierre']}")
print("\nLatencia pareada:", r.get("latencia_pareada_foto"))
print("\n" + open("m5/harness/tabla_delta.md", encoding="utf-8").read())
caida = json.load(open("m5/caida/resultados.json", encoding="utf-8"))["prueba_caida"]
print("PRUEBA DE CAÍDA:", json.dumps(caida, ensure_ascii=False, indent=1))

## 6 · Empaquetar
Descarga `m5_harness.zip` desde *Output* (panel derecho) y extráelo en la raíz del repo: sus rutas empiezan por `m5/`.

In [ ]:
import os, zipfile
with zipfile.ZipFile("/kaggle/working/m5_harness.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["m5/harness", "m5/caida"]:
        for d, _, fs in os.walk(base):
            for f in fs:
                z.write(os.path.join(d, f))
print(zipfile.ZipFile("/kaggle/working/m5_harness.zip").namelist())